# Feature Engineering - ATP Tennis Match Predictor

This notebook builds the feature set used to predict ATP match winners based on findings from the EDA notebook ('01_eda.ipynb'). Rather than using raw match data directly, this notebook transforms it into features that describe the *relative* gap between two players: their ranking, points, and historical performance, since that's what actually determines who's more likely to win a given match.

**Key decisions carried over from EDA:**
- 'Pts_1'/'Pts_2' use '-1' as a missing data placeholder in ~23% of rows (not real NaN)
- 'Odd_1'/'Odd_2' are only reliably available from ~2005 onward
- 'rank_diff' (corr = -0.24) and 'points_diff' (corr = 0.32) both showed real relationships with match outcome and are treated as core features

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/processed/atp_matches_clean.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.shape

(68274, 21)

CSVs don't preserve datetime types, so 'Date' needs to be re-converted with 'pd.to_datetime()' every time the file is reloaded.

In [4]:
df['points_data_missing'] = ((df['Pts_1'] == -1) | (df['Pts_2'] == -1)).astype(int)

df['points_diff'] = np.where(
    df['points_data_missing'] == 1,
    0,
    df['Pts_1'] - df['Pts_2']
)

df['points_data_missing'].mean()

np.float64(0.22888654539063186)

'points_data_missing' is a separate binary column (1 = we don't know the points gap) that the model can learn to use as a signal, letting it learn to trust 'points_diff' less whenever this flag is on.

In [9]:
df['has_odds'] = ((df['Odd_1'] == -1) & (df['Odd_2'] != -1)).astype(int)
df['has_odds'].mean()

np.float64(2.9293728212789642e-05)

'has_odds' is kept as a flag so any odds-based feature can later be built and evaluated only on the subset of rows where it's available

In [10]:
df = df.sort_values('Date').reset_index(drop=True)

The features in this notebook depends on chronological order. 'reset_index(drop=True)' renumbers rows after sorting

In [13]:
p1 = df[['Date', 'Player_1', 'player_1_won', 'Surface']].copy()
p1.columns = ['Date', 'Player', 'Won', 'Surface']

p2 = df[['Date', 'Player_2', 'player_1_won', 'Surface']].copy()
p2['Won'] = 1 - p2['player_1_won']
p2 = p2[['Date', 'Player_2', 'Won', 'Surface']]
p2.columns = ['Date', 'Player', 'Won', 'Surface']

player_matches = pd.concat([p1, p2]).sort_values('Date').reset_index(drop=True)
print(df.shape[0] * 2 == player_matches.shape[0])
player_matches.head()

True


,Date,Player,Won,Surface
0,2000-01-03,Arthurs W.,0,Hard
1,2000-01-03,Ilie A.,0,Hard
2,2000-01-03,Fromberg R.,0,Hard
3,2000-01-03,Henman T.,1,Hard
4,2000-01-03,Henman T.,1,Hard


### Why reshape the data?

Each row in 'df' is one match with two players side by side ('Player_1' and 'Player_2' as separate columns). To calculate something like "Federer's win rate going into this match," it's easier if each player's participation in a match is its own row, rather than being split across two columns depending on which side of the match they happened to be listed on.